# 03 — Synthesize test data

The three synthesizers. Each returns an `EvaluationDataset` of `Golden` seeds;
synthesizer-specific columns live in `Golden.metadata` and flatten out via
`to_pandas()`.

- **Adversarial** runs fully offline (pure pandas).
- **Alignment** needs `transformers` (downloads a T5 paraphraser on first run).
- **RAG** needs `ragas` + a live model.

Each synthesizer is a thin shell over a swappable engine: the constructor takes
the engine, and the `from_*` classmethods build the default one.

## Adversarial — sample / filter a curated bank (offline)

In [ ]:
from llminspector.synthesizer import AdversarialSynthesizer

adv = AdversarialSynthesizer.from_excel(
    "../tests/test_sample/test_adversarialdata.xlsx",
    capability="all",        # or a specific capability / sub-capability
    sample_size=5,
)
adv_dataset = adv.generate()
print("Adversarial goldens:", len(adv_dataset.goldens))

# The extra columns are declared, so the output shape is knowable up front.
print("Metadata columns:", adv.metadata_keys)
adv.to_pandas().head()

## Alignment — tag-augment → HF-T5 paraphrase → perturb

In [ ]:
# from llminspector.synthesizer import AlignmentSynthesizer
#
# align = AlignmentSynthesizer.from_excel(
#     "../tests/test_sample/test_alignmentdata.xlsx",
#     augmentations={"uppercase": ("case_change", 1.0), "typo": ("noise", 0.5)},
#     paraphrase_count=3,
# )
# align_dataset = align.generate()   # downloads the T5 paraphraser on first run
# align.to_pandas().head()

## RAG — ragas `TestsetGenerator` + ground-truth refinement

In [ ]:
# from llminspector.config import AzureSettings
# from llminspector.models import AzureOpenAIEmbedding, AzureOpenAIModel
# from llminspector.synthesizer import RagSynthesizer
#
# settings = AzureSettings.from_env()
# rag = RagSynthesizer.from_documents(
#     model=AzureOpenAIModel(settings),
#     embedding=AzureOpenAIEmbedding(settings),
#     document_dir="path/to/docs",
#     test_size=10,
# )
# rag_dataset = rag.generate()

### Swapping an engine

To replace the algorithm later (a ragas-free RAG backend, a red-teaming attack
source, a custom alignment generator), implement the engine ABC and pass it to
the constructor directly:

```python
RagSynthesizer(YourBackend())
AdversarialSynthesizer(YourAttackSource())
AlignmentSynthesizer(YourAlignmentEngine())
```

No other code changes — `generate() -> EvaluationDataset` is the stable contract.